# Análise exploratória — Tesouro Direto

**Objetivo:** conhecer os dados de preços e taxas dos títulos do Tesouro Direto antes de montar o app web.

**Fonte:** Tesouro Transparente — *Taxas dos Títulos Ofertados pelo Tesouro Direto* (arquivo `precotaxatesourodireto.csv`).

Rode as células **uma de cada vez**, de cima para baixo (clique na célula e aperte **Shift + Enter**). Leia o texto antes de cada código.

## 1. Importar as bibliotecas
- **pandas** (apelido `pd`): lê e organiza a tabela.
- **plotly.express** (apelido `px`): faz os gráficos interativos.

In [28]:
import pandas as pd
import plotly.express as px

## 2. Ler o arquivo
O arquivo do Tesouro segue o padrão brasileiro, então precisamos avisar o pandas:
- `sep=";"` → as colunas são separadas por **ponto e vírgula**;
- `decimal=","` → os números usam **vírgula** como separador decimal (ex.: `13,10`);
- `encoding="latin-1"` → o tipo de codificação de texto do arquivo, para não dar erro com caracteres especiais;
- `"../"` → o arquivo está **uma pasta acima** desta (o notebook fica dentro de `notebooks`).

`df` é o nome da nossa tabela (vem de *DataFrame*). `df.head()` mostra as 5 primeiras linhas.

In [29]:
df = pd.read_csv("../precotaxatesourodireto.csv", sep=";", decimal=",", encoding="latin-1")
df.head()

,Tipo Titulo,Data Vencimento,Data Base,Taxa Compra Manha,Taxa Venda Manha,PU Compra Manha,PU Venda Manha,PU Base Manha
0,Tesouro Selic,01/03/2029,30/10/2026,0.06,0.07,20185.67,20170.66,20170.66
1,Tesouro Selic,01/03/2031,30/10/2026,0.10,0.11,20120.75,20101.76,20101.76
2,Tesouro Selic,01/03/2027,30/10/2026,0.02,0.03,20210.04,20199.05,20199.05
3,Tesouro Selic,01/03/2028,30/10/2026,0.02,0.03,20204.87,20191.85,20191.85
4,Tesouro Prefixado,01/01/2032,30/10/2026,15.38,15.50,479.96,477.14,477.14


## 3. Primeira olhada na tabela
`df.info()` mostra quantas linhas existem, o nome e o **tipo** de cada coluna (texto, número, data) e se há valores vazios.

In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 178072 entries, 0 to 178071
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Tipo Titulo        178072 non-null  str    
 1   Data Vencimento    178072 non-null  str    
 2   Data Base          178072 non-null  str    
 3   Taxa Compra Manha  178072 non-null  float64
 4   Taxa Venda Manha   178072 non-null  float64
 5   PU Compra Manha    178072 non-null  float64
 6   PU Venda Manha     178072 non-null  float64
 7   PU Base Manha      178072 non-null  float64
dtypes: float64(5), str(3)
memory usage: 18.6 MB


`df.describe()` mostra um resumo estatístico das colunas numéricas: média (`mean`), mínimo (`min`), máximo (`max`) e quartis.

In [31]:
df.describe()

,Taxa Compra Manha,Taxa Venda Manha,PU Compra Manha,PU Venda Manha,PU Base Manha
count,178072.000000,178072.000000,178072.000000,178072.000000,178072.000000
mean,7.175959,7.257552,2699.485871,2685.810537,2685.655564
std,4.152083,4.158545,2939.066885,2935.329700,2935.043544
min,-3.530000,-3.410000,0.000000,0.000000,0.000000
25%,5.270000,5.380000,957.210000,954.855000,954.817500
50%,6.820000,6.910000,1768.050000,1754.355000,1754.290000
75%,9.930000,10.010000,3350.202500,3329.532500,3329.307500
max,20.100000,20.130000,20210.040000,20199.050000,20199.050000


## 4. Transformar as datas
As datas vieram como **texto** (ex.: `"30/10/2026"`). Convertendo para o tipo **data**, conseguimos fazer contas e ordenar no tempo.
`format="%d/%m/%Y"` diz que o formato é dia/mês/ano.

In [32]:
df["Data Base"] = pd.to_datetime(df["Data Base"], format="%d/%m/%Y")
df["Data Vencimento"] = pd.to_datetime(df["Data Vencimento"], format="%d/%m/%Y")
df.dtypes

Tipo Titulo                     str
Data Vencimento      datetime64[us]
Data Base            datetime64[us]
Taxa Compra Manha           float64
Taxa Venda Manha            float64
PU Compra Manha             float64
PU Venda Manha              float64
PU Base Manha               float64
dtype: object

## 5. Nova coluna: anos até o vencimento
Quanto tempo falta, em cada dia, até o título vencer? Subtraímos as datas, pegamos os dias (`.dt.days`) e dividimos por 365,25 para ter **anos**.

In [33]:
df["Anos ate Vencimento"] = ((df["Data Vencimento"] - df["Data Base"]).dt.days / 365.25).round(1)
df[["Tipo Titulo", "Data Base", "Data Vencimento", "Anos ate Vencimento"]].head()

,Tipo Titulo,Data Base,Data Vencimento,Anos ate Vencimento
0,Tesouro Selic,2026-10-30,2029-03-01,2.3
1,Tesouro Selic,2026-10-30,2031-03-01,4.3
2,Tesouro Selic,2026-10-30,2027-03-01,0.3
3,Tesouro Selic,2026-10-30,2028-03-01,1.3
4,Tesouro Prefixado,2026-10-30,2032-01-01,5.2


## 6. Quais títulos existem?
`value_counts()` conta quantas linhas há de cada tipo de título.

In [34]:
df["Tipo Titulo"].value_counts()

Tipo Titulo
Tesouro IPCA+ com Juros Semestrais        44226
Tesouro Prefixado                         28270
Tesouro Prefixado com Juros Semestrais    27439
Tesouro Selic                             21509
Tesouro IPCA+                             19008
Tesouro IGPM+ com Juros Semestrais        15824
Tesouro Educa+                            14300
Tesouro Renda+ Aposentadoria Extra         7496
Name: count, dtype: int64

### Agrupar por indexador
A **taxa** não significa a mesma coisa em todos os títulos:
- **Selic** → é só um adicional (spread) **sobre** a Selic;
- **Prefixado** → é a taxa nominal total, já conhecida no dia da compra;
- **IPCA+** (inclui Renda+ e Educa+) → é o ganho **acima da inflação**;
- **IGP-M+** → ganho acima do IGP-M (títulos antigos).

Por isso criamos a coluna **Indexador** — assim nunca comparamos "laranjas com bananas" nos gráficos.

In [35]:
def indexador(nome):
    if "Selic" in nome:
        return "Selic"
    if "Prefixado" in nome:
        return "Prefixado"
    if "IGPM" in nome:
        return "IGP-M+"
    return "IPCA+"  # IPCA+, Renda+ e Educa+

df["Indexador"] = df["Tipo Titulo"].apply(indexador)
df["Indexador"].value_counts()

Indexador
IPCA+        85030
Prefixado    55709
Selic        21509
IGP-M+       15824
Name: count, dtype: int64

## 7. Títulos à venda × títulos só para resgate
Quando um título **não está à venda** naquele dia (só pode ser resgatado), o preço de compra vem **zero**. Vamos contar quantas linhas são assim e criar uma tabela só com os títulos que estavam **à venda**: `df_venda`.

In [36]:
print("Linhas sem preço de compra:", (df["PU Compra Manha"] == 0).sum())

df_venda = df[df["PU Compra Manha"] > 0]
print("Linhas com o título à venda:", len(df_venda))

Linhas sem preço de compra: 49
Linhas com o título à venda: 178023


## 8. Qual período os dados cobrem?

In [37]:
print("Primeira data:", df["Data Base"].min().date())
print("Última data:  ", df["Data Base"].max().date())

Primeira data: 2004-12-31
Última data:   2026-10-30


## 9. Gráfico 1 — Histograma das taxas
O **histograma** mostra **com que frequência** cada faixa de taxa apareceu. Cada cor é um indexador.
Passe o mouse sobre as barras e clique nos nomes da legenda para esconder/mostrar grupos.

In [38]:
fig = px.histogram(
    df_venda,
    x="Taxa Compra Manha",
    color="Indexador",
    barmode="overlay",
    nbins=80,
    title="Distribuição das taxas de compra por indexador",
    labels={"Taxa Compra Manha": "Taxa de compra (% ao ano)"},
)
fig.show()

**O que observar:** as taxas do Selic ficam perto de zero (é só o adicional); as do Prefixado são as maiores (incluem a inflação esperada); as do IPCA+ ficam no meio (são ganho real).

## 10. Gráfico 2 — Dispersão: prazo × taxa no último dia
Pegamos só o **dia mais recente** e colocamos cada título como um ponto: no eixo X, quantos anos faltam para vencer; no eixo Y, a taxa. Isso é parecido com a **curva de juros**: títulos mais longos pagam mais?

In [39]:
ultima_data = df_venda["Data Base"].max()
df_hoje = df_venda[df_venda["Data Base"] == ultima_data]

fig = px.scatter(
    df_hoje,
    x="Anos ate Vencimento",
    y="Taxa Compra Manha",
    color="Indexador",
    hover_name="Tipo Titulo",
    title=f"Prazo × taxa dos títulos à venda em {ultima_data.date()}",
    labels={"Anos ate Vencimento": "Anos até o vencimento", "Taxa Compra Manha": "Taxa de compra (% ao ano)"},
)
fig.show()

## 11. Gráfico 3 — Evolução das taxas ao longo do tempo
Para o gráfico ficar leve, calculamos a **média mensal** da taxa de cada indexador (`freq="ME"` = fim de cada mês).

In [40]:
media_mensal = (
    df_venda
    .groupby([pd.Grouper(key="Data Base", freq="ME"), "Indexador"])["Taxa Compra Manha"]
    .mean()
    .reset_index()
)

fig = px.line(
    media_mensal,
    x="Data Base",
    y="Taxa Compra Manha",
    color="Indexador",
    title="Média mensal da taxa de compra por indexador",
    labels={"Data Base": "Data", "Taxa Compra Manha": "Taxa de compra (% ao ano)"},
)
fig.show()

## 12. Conclusões

### 1. Distribuição das taxas por indexador (histograma)
- **Tesouro Selic:** as taxas se concentram muito perto de 0%, com pequenos ágios e deságios. Como a taxa representa apenas o adicional sobre a Selic, o título acompanha os juros de curto prazo e tem baixíssima oscilação de preço, o que o torna o ativo de liquidez da carteira.
- **Tesouro IPCA+:** concentra-se historicamente entre 4% e 8% ao ano de ganho real. Protege o poder de compra e garante o retorno real contratado para quem leva o título até o vencimento; antes disso, o preço oscila com a marcação a mercado.
- **Tesouro Prefixado:** é o indexador com maior dispersão, entre cerca de 8% e 20% ao ano. Como a taxa é nominal, ela incorpora a inflação esperada e reage aos ciclos de política monetária.

### 2. Estrutura a termo das taxas (prazo × taxa em 30/10/2026)
- **Prefixados (inclinação positiva):** as taxas sobem de cerca de 13,5% ao ano nos vencimentos curtos para cerca de 15,4% entre 5 e 8 anos, e se estabilizam no vencimento mais longo. O mercado exige um prêmio de prazo para emprestar por mais tempo.
- **IPCA+ (curva praticamente plana):** os juros reais ficam entre 7,3% e 7,9% ao ano, com leve queda nos prazos muito longos (vencimentos até 2084). Juros reais acima de 7% são historicamente elevados e indicam condições financeiras restritivas.

### 3. Evolução no tempo e momentos de crise
Os movimentos das taxas coincidem com os principais episódios macroeconômicos do Brasil:
- **2008 (crise financeira global):** pico das taxas prefixadas, num período de forte aversão ao risco.
- **2015–2016 (recessão e crise política):** novo pico, com os prefixados perto de 16% ao ano, acompanhando inflação alta e piora fiscal.
- **2020 (pandemia):** mínimas da série, com prefixados perto de 5% e IPCA+ perto de 2,5% ao ano, no período em que a Selic chegou à mínima histórica de 2% ao ano.
- **2021–2026 (ciclo de alta):** forte subida após o repique inflacionário e o aperto monetário, com os prefixados voltando a cerca de 15% em 2025 e mostrando um recuo no fim de 2026.

### 4. Inflação implícita e qualidade dos dados
- **Inflação implícita:** a distância entre as taxas prefixadas e as de IPCA+ indica, de forma aproximada, a inflação esperada pelo mercado somada a um prêmio de risco de inflação. Essa diferença variou bastante: cerca de 9 pontos em 2005 e 2016, cerca de 2 a 3 pontos em 2020 e cerca de 7 pontos em 2025. *Limitação:* a análise compara médias de títulos com prazos diferentes; o cálculo preciso exige títulos de mesmo vencimento e a fórmula (1 + taxa prefixada) ÷ (1 + taxa real) − 1.
- **Anomalia no IGP-M+ (2020–2021):** a taxa média cai para perto de zero. A contagem de títulos por ano mostra que restavam poucos títulos IGP-M+, já próximos do vencimento — situação em que a taxa fica distorcida. Trata-se de uma limitação da amostra, não de um juro real nulo.